In [1]:
import pandas as pd
import numpy as np
print("pandas version:", pd.__version__)
print("Environment working correctly")


pandas version: 2.2.2
Environment working correctly


In [1]:
!pip install -q datasets==2.21.0

In [2]:
from datasets import load_dataset

dataset = load_dataset(
    "McAuley-Lab/Amazon-Reviews-2023",
    "raw_review_Electronics",
    split="full",
    streaming=True,
    trust_remote_code=True
)

example = next(iter(dataset))
print(example)

{'rating': 3.0, 'title': 'Smells like gasoline! Going back!', 'text': 'First & most offensive: they reek of gasoline so if you are sensitive/allergic to petroleum products like I am you will want to pass on these.  Second: the phone adapter is useless as-is. Mine was not drilled far enough to be able to tighten it into place for my iPhone 12 max. It just slipped & slid all over. Stupid me putting the adapter together first without picking up the binoculars to smell them bc I wasted 15 minutes trying to figure out how to put the adapter together bc it does not come with instructions!  I had to come back here to the website which was a total pain. Third: the tripod is also useless. I would not trust the iOS to hold my $1600 phone nor even a Mattel Barbie for that matter. It’s just inefficient for the job imo.  Third: in order to try to give an honest review I did don gloves & eyewear to check the binoculars out.  They seemed average except for mine seemed to be missing about 10% of the f

In [3]:
import pandas as pd
from datasets import load_dataset

categories = {
    "Electronics": "raw_review_Electronics",
    "Beauty_and_Personal_Care": "raw_review_Beauty_and_Personal_Care",
    "Home_and_Kitchen": "raw_review_Home_and_Kitchen",
    "Software": "raw_review_Software"
}

samples_per_category = 20000
all_rows = []

for category_name, config_name in categories.items():
    print(f"Loading {category_name}...")
    ds = load_dataset(
        "McAuley-Lab/Amazon-Reviews-2023",
        config_name,
        split="full",
        streaming=True,
        trust_remote_code=True
    )

    count = 0
    for row in ds:
        row["category"] = category_name
        all_rows.append(row)
        count += 1
        if count >= samples_per_category:
            break

    print(f"  Collected {count} reviews from {category_name}")

df = pd.DataFrame(all_rows)
print("\nTotal reviews collected:", len(df))
df.head()


Loading Electronics...
  Collected 20000 reviews from Electronics
Loading Beauty_and_Personal_Care...
  Collected 20000 reviews from Beauty_and_Personal_Care
Loading Home_and_Kitchen...
  Collected 20000 reviews from Home_and_Kitchen
Loading Software...
  Collected 20000 reviews from Software

Total reviews collected: 80000


,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase,category
0,3.0,Smells like gasoline! Going back!,First & most offensive: they reek of gasoline ...,[{'small_image_url': 'https://m.media-amazon.c...,B083NRGZMM,B083NRGZMM,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,1658185117948,0,True,Electronics
1,1.0,Didn’t work at all lenses loose/broken.,These didn’t work. Idk if they were damaged in...,[],B07N69T6TM,B07N69T6TM,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,1592678549731,0,True,Electronics
2,5.0,Excellent!,I love these. They even come with a carry case...,[],B01G8JO5F2,B01G8JO5F2,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,1523093017534,0,True,Electronics
3,5.0,Great laptop backpack!,I was searching for a sturdy backpack for scho...,[],B001OC5JKY,B001OC5JKY,AGGZ357AO26RQZVRLGU4D4N52DZQ,1290278495000,18,True,Electronics
4,5.0,Best Headphones in the Fifties price range!,I've bought these headphones three times becau...,[],B013J7WUGC,B07CJYMRWM,AG2L7H23R5LLKDKLBEF2Q3L2MVDA,1676601581238,0,True,Electronics


In [4]:
df.to_parquet("amazon_reviews_raw.parquet", index=False)
print("Saved. Shape:", df.shape)

Saved. Shape: (80000, 11)


In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
import shutil
shutil.copy("amazon_reviews_raw.parquet", "/content/drive/MyDrive/amazon_reviews_raw.parquet")
print("Saved to Drive")

Saved to Drive


In [7]:
from google.colab import files
files.download("amazon_reviews_raw.parquet")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [8]:
print(df.shape)
print(df['category'].value_counts())
print(df['rating'].value_counts())
print(df.isnull().sum())

(80000, 11)
category
Electronics                 20000
Beauty_and_Personal_Care    20000
Home_and_Kitchen            20000
Software                    20000
Name: count, dtype: int64
rating
5.0    48942
4.0    13814
3.0     7077
1.0     6492
2.0     3675
Name: count, dtype: int64
rating               0
title                0
text                 0
images               0
asin                 0
parent_asin          0
user_id              0
timestamp            0
helpful_vote         0
verified_purchase    0
category             0
dtype: int64


In [9]:
df = df.dropna(subset=['text'])
df = df[df['text'].str.strip() != '']
print("Shape after dropping empty text:", df.shape)

Shape after dropping empty text: (79985, 11)


In [10]:
import re

def clean_text(text):
    text = str(text)
    text = re.sub(r'<.*?>', ' ', text)          # remove HTML tags
    text = re.sub(r'http\S+|www\S+', ' ', text)  # remove URLs
    text = re.sub(r'\s+', ' ', text)             # collapse whitespace
    text = text.strip()
    return text

df['clean_text'] = df['text'].apply(clean_text)
df[['text', 'clean_text']].head()

,text,clean_text
0,First & most offensive: they reek of gasoline ...,First & most offensive: they reek of gasoline ...
1,These didn’t work. Idk if they were damaged in...,These didn’t work. Idk if they were damaged in...
2,I love these. They even come with a carry case...,I love these. They even come with a carry case...
3,I was searching for a sturdy backpack for scho...,I was searching for a sturdy backpack for scho...
4,I've bought these headphones three times becau...,I've bought these headphones three times becau...


In [11]:
df['text_length'] = df['clean_text'].str.split().str.len()
print(df['text_length'].describe())

df = df[df['text_length'] >= 3]
print("Shape after removing very short reviews:", df.shape)

count    79985.000000
mean        64.340801
std         98.483768
min          1.000000
25%         11.000000
50%         30.000000
75%         78.000000
max       3929.000000
Name: text_length, dtype: float64
Shape after removing very short reviews: (74545, 13)


In [12]:
def label_sentiment(rating):
    if rating <= 2:
        return 'negative'
    elif rating == 3:
        return 'neutral'
    else:
        return 'positive'

df['sentiment'] = df['rating'].apply(label_sentiment)
print(df['sentiment'].value_counts())
print(df['sentiment'].value_counts(normalize=True))

sentiment
positive    58302
negative     9581
neutral      6662
Name: count, dtype: int64
sentiment
positive    0.782105
negative    0.128526
neutral     0.089369
Name: proportion, dtype: float64


In [13]:
df_clean = df[['asin', 'category', 'rating', 'title', 'clean_text', 'sentiment',
               'helpful_vote', 'verified_purchase', 'timestamp']].copy()

df_clean.to_parquet("amazon_reviews_clean.parquet", index=False)
print("Saved. Shape:", df_clean.shape)
df_clean.head()

Saved. Shape: (74545, 9)


,asin,category,rating,title,clean_text,sentiment,helpful_vote,verified_purchase,timestamp
0,B083NRGZMM,Electronics,3.0,Smells like gasoline! Going back!,First & most offensive: they reek of gasoline ...,neutral,0,True,1658185117948
1,B07N69T6TM,Electronics,1.0,Didn’t work at all lenses loose/broken.,These didn’t work. Idk if they were damaged in...,negative,0,True,1592678549731
2,B01G8JO5F2,Electronics,5.0,Excellent!,I love these. They even come with a carry case...,positive,0,True,1523093017534
3,B001OC5JKY,Electronics,5.0,Great laptop backpack!,I was searching for a sturdy backpack for scho...,positive,18,True,1290278495000
4,B013J7WUGC,Electronics,5.0,Best Headphones in the Fifties price range!,I've bought these headphones three times becau...,positive,0,True,1676601581238


In [14]:
import shutil
shutil.copy("amazon_reviews_clean.parquet", "/content/drive/MyDrive/amazon_reviews_clean.parquet")

'/content/drive/MyDrive/amazon_reviews_clean.parquet'

In [16]:
from google.colab import files
files.download("amazon_reviews_clean.parquet")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>